# Tutorial 04 — Instruction tuning and parameter-efficient adaptation

[Lecture notes](../lectures/04_adaptation.md) · [Exercise sheet](../exercises/04_adaptation.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Fit a low-rank adapter to a frozen linear model and inspect response-only label masking.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Construct an adaptation task

Synthetic targets differ from the base model by a rank-two residual. This is a linear LoRA mechanics experiment, not language-model fine-tuning.

Before running: predict the output or the invariant being checked.


In [2]:
n,din,dout,r=128,12,8,2
X=rng.normal(size=(n,din)); Xtest=rng.normal(size=(64,din))
Wbase=rng.normal(0,.2,(din,dout))
true_delta=rng.normal(0,.2,(din,r))@rng.normal(0,.2,(r,dout))
Y=X@(Wbase+true_delta); Ytest=Xtest@(Wbase+true_delta)
A=rng.normal(0,.1,(din,r)); B=np.zeros((r,dout)); scale=1.0
frozen=Wbase.copy()
def mse(x,y): return np.mean((x@(Wbase+scale*A@B)-y)**2)
print("base test MSE",mse(Xtest,Ytest),"adapter/full parameters",r*(din+dout),din*dout)


base test MSE 0.05412808238153114 adapter/full parameters 40 96


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Optimize only adapter parameters

Calculate both gradients before either update. Inspect the first-step gradient of A.

Before running: predict the output or the invariant being checked.


In [3]:
initial=mse(Xtest,Ytest)
for step in range(401):
    err=X@(Wbase+scale*A@B)-Y
    G=2*X.T@err/err.size
    dA=scale*G@B.T; dB=scale*A.T@G
    if step==0: assert np.allclose(dA,0) and np.linalg.norm(dB)>0
    A-=.5*dA; B-=.5*dB
    if step%100==0: print(step,"train",mse(X,Y),"test",mse(Xtest,Ytest))
assert np.array_equal(Wbase,frozen)
assert mse(Xtest,Ytest)<initial


0 train 0.06146098485189066 test 0.05386557729664078
100 train 0.0026530153745442002 test 0.004125273080141428
200 train 9.402783747706043e-06 test 1.9301633920292944e-05
300 train 5.799000950589515e-08 test 1.3488593607492863e-07
400 train 5.892575182561166e-10 test 1.3255859644546835e-09


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Inspect response-only supervision

The causal model shifts labels internally in the optional HF extension. Here we inspect only which original token positions carry labels.

Before running: predict the output or the invariant being checked.


In [4]:
# IDs 10/11/12 are an illustrative prompt; 20/21 are response; 2 is EOS; 0 is padding.
input_ids=np.array([[10,11,12,20,21,2,0,0]])
attention_mask=np.array([[1,1,1,1,1,1,0,0]])
labels=input_ids.copy(); labels[:,:3]=-100; labels[attention_mask==0]=-100
print("inputs",input_ids,"labels",labels)
assert np.array_equal(labels,np.array([[-100,-100,-100,20,21,2,-100,-100]]))
assert (input_ids[:,:3]!=-100).all()


inputs [[10 11 12 20 21  2  0  0]] labels [[-100 -100 -100   20   21    2 -100 -100]]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why initialize only one LoRA factor to zero?
2. Should prompt tokens be removed from the input for response-only SFT?
3. What must be saved alongside an adapter to reproduce inference?


## Extension and submission

Run `extensions/hf_lora.py` for a small actual SFT experiment; inspect the printed trainable parameters and token masks before interpreting loss.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
